In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\jiyam\OneDrive\Desktop\Customer Retention & Churn Analysis\Telco-Customer-Churn.csv")


df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
#Standardize Column Names
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [3]:
#Handle Missing Values
df.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [7]:
print(df.columns)


Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')


In [8]:
#Fix numeric conversion
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')

In [10]:
#Handle missing values created during conversion
df['totalcharges'] = df['totalcharges'].fillna(df['totalcharges'].median())

In [11]:
#For categorical columns
df['internetservice'] = df['internetservice'].fillna(df['internetservice'].mode()[0])

In [12]:
#Convert Target Variable (Churn)
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})

In [13]:
#Clean Categorical Columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.lower().str.strip()

In [14]:
#Fix Special Values
cols = [
    'multiplelines','onlinesecurity','onlinebackup',
    'deviceprotection','techsupport','streamingtv','streamingmovies'
]

for col in cols:
    df[col] = df[col].replace('no internet service', 'no')
    df[col] = df[col].replace('no phone service', 'no')

In [15]:
#Convert Binary Columns (Optional but good)
binary_cols = ['partner','dependents','phoneservice','paperlessbilling']

for col in binary_cols:
    df[col] = df[col].map({'yes':1, 'no':0})

In [16]:
#Remove Duplicates
df.duplicated().sum()

df.drop_duplicates(inplace=True)

In [17]:
#Outlier Check (Important for CLV)
df[['monthlycharges','totalcharges']].describe()

,monthlycharges,totalcharges
count,7043.000000,7043.000000
mean,64.761692,2281.916928
std,30.090047,2265.270398
min,18.250000,18.800000
25%,35.500000,402.225000
50%,70.350000,1397.475000
75%,89.850000,3786.600000
max,118.750000,8684.800000


In [18]:
#Tenure groups (cohort analysis)
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0,6,12,24,48,100],
    labels=['0-6','6-12','1-2yr','2-4yr','4+yr']
)

In [19]:
#Customer Lifetime Value (CLV proxy)
df['clv'] = df['monthlycharges'] * df['tenure']

In [20]:
#High/Low spender segment
df['spend_category'] = pd.cut(
    df['monthlycharges'],
    bins=[0,30,70,100,200],
    labels=['low','medium','high','very_high']
)

In [21]:
#Final Validation
df.info()
df.isnull().sum()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerid        7043 non-null   object  
 1   gender            7043 non-null   object  
 2   seniorcitizen     7043 non-null   int64   
 3   partner           7043 non-null   int64   
 4   dependents        7043 non-null   int64   
 5   tenure            7043 non-null   int64   
 6   phoneservice      7043 non-null   int64   
 7   multiplelines     7043 non-null   object  
 8   internetservice   7043 non-null   object  
 9   onlinesecurity    7043 non-null   object  
 10  onlinebackup      7043 non-null   object  
 11  deviceprotection  7043 non-null   object  
 12  techsupport       7043 non-null   object  
 13  streamingtv       7043 non-null   object  
 14  streamingmovies   7043 non-null   object  
 15  contract          7043 non-null   object  
 16  paperlessbilling  7043 n

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn,tenure_group,clv,spend_category
0,7590-vhveg,female,0,1,0,1,0,no,dsl,no,...,no,month-to-month,1,electronic check,29.85,29.85,0,0-6,29.85,low
1,5575-gnvde,male,0,0,0,34,1,no,dsl,yes,...,no,one year,0,mailed check,56.95,1889.50,0,2-4yr,1936.30,medium
2,3668-qpybk,male,0,0,0,2,1,no,dsl,yes,...,no,month-to-month,1,mailed check,53.85,108.15,1,0-6,107.70,medium
3,7795-cfocw,male,0,0,0,45,0,no,dsl,yes,...,no,one year,0,bank transfer (automatic),42.30,1840.75,0,2-4yr,1903.50,medium
4,9237-hqitu,female,0,0,0,2,1,no,fiber optic,no,...,no,month-to-month,1,electronic check,70.70,151.65,1,0-6,141.40,high


In [23]:
#Save Clean Dataset
df.to_csv("cleaned_churn_data.csv", index=False)


In [24]:
#Overall churn rate
df['churn'].mean()

np.float64(0.2653698707936959)

In [25]:
#Churn by contract
df.groupby('contract')['churn'].mean()

contract
month-to-month    0.427097
one year          0.112695
two year          0.028319
Name: churn, dtype: float64

In [26]:
#Churn by internet service
df.groupby('internetservice')['churn'].mean()

internetservice
dsl            0.189591
fiber optic    0.418928
no             0.074050
Name: churn, dtype: float64

In [27]:
#Churn by payment method
df.groupby('paymentmethod')['churn'].mean()

paymentmethod
bank transfer (automatic)    0.167098
credit card (automatic)      0.152431
electronic check             0.452854
mailed check                 0.191067
Name: churn, dtype: float64

In [28]:
#Tenure vs churn
df.groupby('tenure')['churn'].mean()

tenure
0     0.000000
1     0.619902
2     0.516807
3     0.470000
4     0.471591
        ...   
68    0.090000
69    0.084211
70    0.092437
71    0.035294
72    0.016575
Name: churn, Length: 73, dtype: float64

In [29]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0,6,12,24,48,100],
    labels=['0-6','6-12','1-2yr','2-4yr','4+yr']
)

In [30]:
df.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn',
       'tenure_group', 'clv', 'spend_category'],
      dtype='object')

In [31]:
df['tenure_group'].unique()

['0-6', '2-4yr', '6-12', '1-2yr', '4+yr', NaN]
Categories (5, object): ['0-6' < '6-12' < '1-2yr' < '2-4yr' < '4+yr']

In [32]:
df.select_dtypes(include='object').head()

,customerid,gender,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paymentmethod
0,7590-vhveg,female,no,dsl,no,yes,no,no,no,no,month-to-month,electronic check
1,5575-gnvde,male,no,dsl,yes,no,yes,no,no,no,one year,mailed check
2,3668-qpybk,male,no,dsl,yes,yes,no,no,no,no,month-to-month,mailed check
3,7795-cfocw,male,no,dsl,yes,no,yes,yes,no,no,one year,bank transfer (automatic)
4,9237-hqitu,female,no,fiber optic,no,no,no,no,no,no,month-to-month,electronic check


In [34]:
df.to_csv("cleaned_customer_churn.csv", index=False)

In [35]:
print(df.columns)

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn',
       'tenure_group', 'clv', 'spend_category'],
      dtype='object')


In [48]:
df['tenure_group'].unique()

['0-6', '2-4yr', '6-12', '1-2yr', '4+yr', NaN]
Categories (5, object): ['0-6' < '6-12' < '1-2yr' < '2-4yr' < '4+yr']

In [37]:
df[['tenure', 'tenure_group']].head(10)

,tenure,tenure_group
0,1,0-6
1,34,2-4yr
2,2,0-6
3,45,2-4yr
4,2,0-6
5,8,6-12
6,22,1-2yr
7,10,6-12
8,28,2-4yr
9,62,4+yr


In [38]:
df['tenure_group'].value_counts()

tenure_group
4+yr     2239
2-4yr    1594
0-6      1470
1-2yr    1024
6-12      705
Name: count, dtype: int64

In [52]:
df.drop(columns=['tenure_group'], inplace=True)



In [50]:
df.groupby('tenure')['churn'].mean()

tenure
0     0.000000
1     0.619902
2     0.516807
3     0.470000
4     0.471591
        ...   
68    0.090000
69    0.084211
70    0.092437
71    0.035294
72    0.016575
Name: churn, Length: 73, dtype: float64

In [53]:
df.to_csv("cleaned_customer_churn_final.csv", index=False)

In [54]:
#Overall Churn Rate
df['churn'].value_counts(normalize=True) * 100

churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

In [55]:
#Churn by Contract Type
pd.crosstab(df['contract'], df['churn'], normalize='index') * 100

churn,0,1
contract,,
month-to-month,57.290323,42.709677
one year,88.730482,11.269518
two year,97.168142,2.831858


In [56]:
#Churn by Payment Method
pd.crosstab(df['paymentmethod'], df['churn'], normalize='index') * 100

churn,0,1
paymentmethod,,
bank transfer (automatic),83.290155,16.709845
credit card (automatic),84.756899,15.243101
electronic check,54.714588,45.285412
mailed check,80.893300,19.106700


In [57]:
#Churn by Internet Service
pd.crosstab(df['internetservice'], df['churn'], normalize='index') * 100

churn,0,1
internetservice,,
dsl,81.040892,18.959108
fiber optic,58.107235,41.892765
no,92.595020,7.404980


In [58]:
#Average Tenure by Churn
df.groupby('churn')['tenure'].mean()

churn
0    37.569965
1    17.979133
Name: tenure, dtype: float64

In [59]:
#Average CLV by Churn
df.groupby('churn')['clv'].mean()

churn
0    2549.770883
1    1531.608828
Name: clv, dtype: float64

In [60]:
#Senior Citizen Churn
pd.crosstab(df['seniorcitizen'], df['churn'], normalize='index') * 100

churn,0,1
seniorcitizen,,
0,76.393832,23.606168
1,58.318739,41.681261


In [61]:
#Gender Churn
pd.crosstab(df['gender'], df['churn'], normalize='index') * 100

churn,0,1
gender,,
female,73.079128,26.920872
male,73.839662,26.160338


In [62]:
#Numerical Summary
df[['tenure','monthlycharges','totalcharges','clv']].describe()

,tenure,monthlycharges,totalcharges,clv
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2281.916928,2279.581350
std,24.559481,30.090047,2265.270398,2264.729447
min,0.000000,18.250000,18.800000,0.000000
25%,9.000000,35.500000,402.225000,394.000000
50%,29.000000,70.350000,1397.475000,1393.600000
75%,55.000000,89.850000,3786.600000,3786.100000
max,72.000000,118.750000,8684.800000,8550.000000


In [63]:
#Correlation
df[['tenure','monthlycharges','totalcharges','clv','churn']].corr()


,tenure,monthlycharges,totalcharges,clv,churn
tenure,1.000000,0.247900,0.825464,0.826568,-0.352229
monthlycharges,0.247900,1.000000,0.650864,0.651566,0.193356
totalcharges,0.825464,0.650864,1.000000,0.999263,-0.199037
clv,0.826568,0.651566,0.999263,1.000000,-0.198514
churn,-0.352229,0.193356,-0.199037,-0.198514,1.000000
